In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
len(np.unique(y))

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader
import torch.nn.functional as F

X_train=torch.tensor(X_train,dtype=torch.float32)
X_test=torch.tensor(X_test,dtype=torch.float32)
y_train=torch.tensor(y_train,dtype=torch.float32)
y_test=torch.tensor(y_test,dtype=torch.float32)

In [ ]:
# 2. Create TensorDataset objects

train_dataset=TensorDataset(X_train,y_train)
test_dataset=TensorDataset(X_test,y_test)

In [ ]:
# 3. Create DataLoaders

train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,drop_last=True) # i use drop last if last batch not even like if its  [31] for example i drop it i just want fill it all with 32
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)

In [ ]:
# 4. Print shape of one batch
x_tr,y_tr=next(iter(train_loader))
x_tt,y_tt=next(iter(test_loader))

print(x_tr.shape,y_tr.shape)
print(x_tt.shape,y_tt.shape)

In [ ]:
# 5. Display sample images
import matplotlib.pyplot as plt

images, labels = next(iter(train_loader))

# Display the first 6 images in the batch
plt.figure(figsize=(8, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1)

    # Convert from (C, H, W) to (H, W, C) for matplotlib
    img = images[i].permute(1, 2, 0)

    plt.imshow(img)
    plt.title(f"Label: {labels[i].item()}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Task 1: Write your model class here:
class NN5Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super(NN5Layer, self).__init__()

        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2=nn.Linear(hidden_dim,hidden_dim)
        self.layer3=nn.Linear(hidden_dim,hidden_dim)
        self.output=nn.Linear(hidden_dim,output_dim)

        self.relu = nn.ReLU()

    # Defines how input data flows through the network
    def forward(self, x):
      x=self.relu(self.layer1(x))
      x=self.relu(self.layer2(x))
      x=self.relu(self.layer3(x))
      return self.output(x)




In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, optimizer, criterion, train_loader, device=None):
    # Set the model to training mode
    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.view(X_batch.size(0), -1)
        X_batch=X_batch.to(torch.float16)

        y_batch = y_batch.to(torch.float16)


        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    return avg_loss

In [ ]:
# Task 3: Write your validation loop here:
def validate(model, criterion, test_loader, device=None):
    model.eval()

    running_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch=X_batch.to(torch.float16)

            X_batch = X_batch.view(X_batch.size(0), -1)

            y_batch = y_batch.to(torch.float16)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item()


        avg_loss = running_loss / len(test_loader)

    return avg_loss

In [ ]:
# Task 4: Define device, model, loss, optimizer:
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu") i do not need the


input_dim = 3 * 36 * 36

hidden_dim = 128

output_dim = 32

model = NN5Layer(input_dim, hidden_dim, output_dim)

print("Model Architecture:\n")
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params}")

num_epochs = 20
learning_rate = 0.01

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), learning_rate)

In [ ]:
# Task 5: Start training for 20 epochs:
train_losses = []
val_losses = []
val_accuracies = []

print('Starting Training...')
for epoch in range(num_epochs):
    # Train one epoch
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device=None)

    # Validate
    val_loss = validate(model, criterion, test_loader, device=None)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')
    #print(f'Train Loss: {train_loss:.4f}')

print('Training Complete!')


# there is an error with data dtype even i set all float32 but i couldent solve it i tried


In [ ]:
# Task 1: Write your code here:

plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here: